STAGE 4.7
# OPENVINO CONVERSION

In [2]:
# ============================================================
# CISLR RESEARCH — STAGE 4.7
# OPENVINO CONVERSION
# ============================================================
#
# Selected model:
#   MobileNetV3-Large
#
# Input:
#   Stage 4.4 best MobileNetV3-Large checkpoint
#
# Output:
#   OpenVINO IR FP32 model
#   OpenVINO conversion metadata
#   PyTorch vs OpenVINO numerical validation
#
# IMPORTANT:
#   - NO RETRAINING
#   - NO VIDEO DECODING
#   - NO Stage 4.3 rerun
#   - NO Stage 4.4 rerun
#   - NO Stage 4.5 rerun
#   - NO Stage 4.6 rerun
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

from pathlib import Path
import json
import hashlib
import time
import sys
import platform

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision

import openvino as ov


# ============================================================
# 1. PROJECT PATHS
# ============================================================

ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

MODEL_DIR = (
    ROOT /
    "models" /
    "stage4_4" /
    "mobilenet_v3_large_v2"
)

CHECKPOINT_PATH = (
    MODEL_DIR /
    "mobilenet_v3_large_stage4_4_v2_best.pt"
)

STAGE46_DIR = (
    ROOT /
    "audit" /
    "stage4_6_model_comparison"
)

STAGE47_DIR = (
    ROOT /
    "audit" /
    "stage4_7_openvino_conversion"
)

STAGE47_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FP32_DIR = (
    STAGE47_DIR /
    "fp32"
)

FP32_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 80)
print("CISLR — STAGE 4.7 OPENVINO CONVERSION")
print("=" * 80)

print()
print("Project root:")
print(ROOT)

print()
print("Selected model:")
print("MobileNetV3-Large")

print()
print("Checkpoint:")
print(CHECKPOINT_PATH)

print()
print("Output:")
print(STAGE47_DIR)

print()


# ============================================================
# 2. ENVIRONMENT INFORMATION
# ============================================================

print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)

print(
    "Python:",
    sys.version
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Torchvision:",
    torchvision.__version__
)

print(
    "OpenVINO:",
    ov.__version__
)

print(
    "Platform:",
    platform.platform()
)

print()


# ============================================================
# 3. VERIFY STAGE 4.6 SELECTION
# ============================================================

stage46_csv = (
    STAGE46_DIR /
    "stage4_6_model_comparison_final.csv"
)

stage46_json = (
    STAGE46_DIR /
    "stage4_6_model_selection_report.json"
)

assert stage46_csv.exists(), (
    f"Stage 4.6 comparison not found:\n{stage46_csv}"
)

assert stage46_json.exists(), (
    f"Stage 4.6 selection report not found:\n{stage46_json}"
)


stage46_df = pd.read_csv(
    stage46_csv
)

with open(
    stage46_json,
    "r"
) as f:

    stage46_report = json.load(f)


selected_model = (
    stage46_report["selected_model"]
)

print(
    "Stage 4.6 selected model:",
    selected_model
)

assert selected_model == "MobileNetV3-Large", (
    "Stage 4.6 does not select MobileNetV3-Large. "
    "Do not continue with this cell."
)

print(
    "Stage 4.6 selection verified."
)

print()


# ============================================================
# 4. VERIFY CHECKPOINT
# ============================================================

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found:\n{CHECKPOINT_PATH}"
)

checkpoint_size_mb = (
    CHECKPOINT_PATH.stat().st_size /
    (1024 ** 2)
)

print("=" * 80)
print("CHECKPOINT")
print("=" * 80)

print(
    f"Path : {CHECKPOINT_PATH}"
)

print(
    f"Size : {checkpoint_size_mb:.3f} MB"
)

print()


# ============================================================
# 5. LOAD CHECKPOINT
# ============================================================

print(
    "Loading PyTorch checkpoint..."
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)


print(
    "Checkpoint object type:",
    type(checkpoint)
)

print()


# ============================================================
# 6. INSPECT CHECKPOINT STRUCTURE
# ============================================================

if isinstance(
    checkpoint,
    dict
):

    print(
        "Checkpoint dictionary keys:"
    )

    for key in checkpoint.keys():

        print(
            f"  - {key}"
        )

else:

    print(
        "Checkpoint is not a dictionary."
    )

print()


# ============================================================
# 7. EXTRACT STATE DICTIONARY
# ============================================================

state_dict = None

if isinstance(
    checkpoint,
    dict
):

    possible_state_keys = [
        "model_state_dict",
        "state_dict",
        "model",
        "net",
        "weights",
    ]

    for key in possible_state_keys:

        if key in checkpoint:

            candidate = checkpoint[key]

            if isinstance(
                candidate,
                dict
            ):

                state_dict = candidate

                print(
                    f"Using state dictionary from key: "
                    f"{key}"
                )

                break


if state_dict is None:

    if isinstance(
        checkpoint,
        dict
    ):

        # Some checkpoints are directly state_dict objects.
        tensor_values = [
            v
            for v in checkpoint.values()
            if torch.is_tensor(v)
        ]

        if len(tensor_values) > 0:

            state_dict = checkpoint

            print(
                "Checkpoint itself appears to be "
                "a state_dict."
            )


if state_dict is None:

    raise RuntimeError(
        "Could not identify a PyTorch state_dict "
        "inside the checkpoint."
    )


print()


# ============================================================
# 8. REMOVE COMMON PREFIXES
# ============================================================

clean_state_dict = {}

for key, value in state_dict.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):

        new_key = new_key[
            len("module.") :
        ]

    if new_key.startswith(
        "_orig_mod."
    ):

        new_key = new_key[
            len("_orig_mod.") :
        ]

    clean_state_dict[
        new_key
    ] = value


state_dict = clean_state_dict


print(
    f"State-dict parameters: "
    f"{len(state_dict):,}"
)

print()


# ============================================================
# 9. IDENTIFY CLASSIFIER OUTPUT SIZE
# ============================================================
#
# CISLR has 4764 valid classes.
#
# MobileNetV3-Large torchvision:
#
#   features
#      ↓
#   avgpool
#      ↓
#   flatten
#      ↓
#   classifier
#
# Standard torchvision classifier:
#
#   Linear(960, 1280)
#   Hardswish
#   Dropout
#   Linear(1280, num_classes)
#
# ============================================================

classifier_weight_keys = [
    key
    for key in state_dict.keys()
    if (
        "classifier" in key
        and key.endswith("weight")
        and state_dict[key].ndim == 2
    )
]


print(
    "Classifier weight candidates:"
)

for key in classifier_weight_keys:

    print(
        f"  {key}: "
        f"{tuple(state_dict[key].shape)}"
    )

print()


# Find final classifier weight.
# It should have shape [4764, 1280].

final_classifier_key = None

for key in classifier_weight_keys:

    shape = tuple(
        state_dict[key].shape
    )

    if shape[0] == 4764:

        final_classifier_key = key
        break


assert final_classifier_key is not None, (
    "Could not find the final 4764-class MobileNet "
    "classifier weight."
)


num_classes = int(
    state_dict[
        final_classifier_key
    ].shape[0]
)

print(
    "Detected classifier output classes:",
    num_classes
)

assert num_classes == 4764, (
    f"Expected 4764 classes, got {num_classes}"
)

print()


# ============================================================
# 10. CREATE EXACT TORCHVISION ARCHITECTURE
# ============================================================

print("=" * 80)
print("CREATING MOBILEV3-LARGE ARCHITECTURE")
print("=" * 80)

model = torchvision.models.mobilenet_v3_large(
    weights=None
)


# Replace final classifier with the exact 4764-class head.
#
# torchvision MobileNetV3-Large:
#
# classifier[-1]:
#     Linear(1280, 1000)
#
# We need:
#     Linear(1280, 4764)

model.classifier[-1] = nn.Linear(
    1280,
    num_classes
)


print(
    model.classifier
)

print()


# ============================================================
# 11. LOAD TRAINED WEIGHTS
# ============================================================

print(
    "Loading trained Stage 4.4 weights..."
)

missing_keys, unexpected_keys = (
    model.load_state_dict(
        state_dict,
        strict=False
    )
)


print(
    f"Missing keys    : {len(missing_keys)}"
)

print(
    f"Unexpected keys : {len(unexpected_keys)}"
)


if len(missing_keys) > 0:

    print()
    print(
        "Missing keys:"
    )

    for key in missing_keys[:20]:

        print(
            f"  {key}"
        )


if len(unexpected_keys) > 0:

    print()
    print(
        "Unexpected keys:"
    )

    for key in unexpected_keys[:20]:

        print(
            f"  {key}"
        )


assert len(missing_keys) == 0, (
    "Checkpoint loading produced missing keys."
)

assert len(unexpected_keys) == 0, (
    "Checkpoint loading produced unexpected keys."
)


print()
print(
    "Checkpoint loaded successfully."
)

print()


# ============================================================
# 12. MODEL EVALUATION MODE
# ============================================================

model.eval()

for parameter in model.parameters():

    parameter.requires_grad = False


# ============================================================
# 13. PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 80)
print("MODEL INFORMATION")
print("=" * 80)

print(
    f"Total parameters    : {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)

print(
    f"Output classes      : {num_classes}"
)

print()


# ============================================================
# 14. EXAMPLE INPUT
# ============================================================
#
# Stage 4.3 used:
#
#   batch = 1
#   channels = 3
#   height = 224
#   width = 224
#
# Therefore we preserve exactly:
#
#   [1, 3, 224, 224]
#
# ============================================================

example_input = torch.randn(
    1,
    3,
    224,
    224,
    dtype=torch.float32
)


print(
    "Example input shape:",
    tuple(example_input.shape)
)

print(
    "Example input dtype:",
    example_input.dtype
)

print()


# ============================================================
# 15. PYTORCH REFERENCE OUTPUT
# ============================================================

print("=" * 80)
print("PYTORCH REFERENCE INFERENCE")
print("=" * 80)

with torch.inference_mode():

    torch_output = model(
        example_input
    )


print(
    "PyTorch output shape:",
    tuple(torch_output.shape)
)

print(
    "PyTorch output dtype:",
    torch_output.dtype
)

assert tuple(
    torch_output.shape
) == (
    1,
    4764
)

print()


# ============================================================
# 16. OPENVINO CONVERSION
# ============================================================
#
# Current OpenVINO recommended PyTorch conversion:
#
#   ov.convert_model(
#       model,
#       example_input=example_input
#   )
#
# The example_input is deliberately supplied so the PyTorch
# model graph is traced with the exact deployment shape.
#
# ============================================================

print("=" * 80)
print("OPENVINO CONVERSION")
print("=" * 80)

print(
    "Converting MobileNetV3-Large..."
)

conversion_start = time.perf_counter()


ov_model = ov.convert_model(
    model,
    example_input=example_input,
    verbose=False
)


conversion_time = (
    time.perf_counter()
    -
    conversion_start
)


print(
    f"Conversion time: "
    f"{conversion_time:.3f} seconds"
)

print()


# ============================================================
# 17. INSPECT OPENVINO INPUT
# ============================================================

print("=" * 80)
print("OPENVINO MODEL INPUT")
print("=" * 80)

ov_inputs = ov_model.inputs

print(
    f"Number of inputs: "
    f"{len(ov_inputs)}"
)

for i, inp in enumerate(
    ov_inputs
):

    print(
        f"Input {i}:"
    )

    print(
        f"  Name   : {inp.get_any_name()}"
    )

    print(
        f"  Shape  : {inp.partial_shape}"
    )

    print(
        f"  Type   : {inp.element_type}"
    )

print()


# ============================================================
# 18. INSPECT OPENVINO OUTPUT
# ============================================================

print("=" * 80)
print("OPENVINO MODEL OUTPUT")
print("=" * 80)

ov_outputs = ov_model.outputs

print(
    f"Number of outputs: "
    f"{len(ov_outputs)}"
)

for i, output in enumerate(
    ov_outputs
):

    print(
        f"Output {i}:"
    )

    names = output.get_names()
    if names:
        name = next(iter(names))
        print(f"  Name  : {name}")
    else:
        print("  Name  : <unnamed>")

    print(
        f"  Shape : {output.partial_shape}"
    )

    print(
        f"  Type  : {output.element_type}"
    )

print()


# ============================================================
# 19. VERIFY OPENVINO OUTPUT SHAPE
# ============================================================

assert len(ov_inputs) == 1, (
    "Expected exactly one model input."
)

assert len(ov_outputs) == 1, (
    "Expected exactly one model output."
)


ov_output_shape = (
    ov_outputs[0]
    .partial_shape
)


print(
    "OpenVINO output shape:",
    ov_output_shape
)

print()


# ============================================================
# 20. SAVE FP32 OPENVINO IR
# ============================================================
#
# This is the canonical Stage 4.7 FP32 conversion.
#
# It produces:
#
#   mobilenet_v3_large_stage4_7_fp32.xml
#   mobilenet_v3_large_stage4_7_fp32.bin
#
# ============================================================

FP32_XML = (
    FP32_DIR /
    "mobilenet_v3_large_stage4_7_fp32.xml"
)


print("=" * 80)
print("SAVING OPENVINO FP32 IR")
print("=" * 80)

save_start = time.perf_counter()


ov.save_model(
    ov_model,
    str(FP32_XML),
    compress_to_fp16=False
)


save_time = (
    time.perf_counter()
    -
    save_start
)


print(
    f"Saved in {save_time:.3f} seconds"
)

print(
    f"XML: {FP32_XML}"
)

print(
    f"BIN: {FP32_XML.with_suffix('.bin')}"
)

print()


# ============================================================
# 21. VERIFY FILES
# ============================================================

FP32_BIN = (
    FP32_XML.with_suffix(
        ".bin"
    )
)


assert FP32_XML.exists(), (
    f"Missing XML:\n{FP32_XML}"
)

assert FP32_BIN.exists(), (
    f"Missing BIN:\n{FP32_BIN}"
)


xml_size_mb = (
    FP32_XML.stat().st_size /
    (1024 ** 2)
)

bin_size_mb = (
    FP32_BIN.stat().st_size /
    (1024 ** 2)
)

total_ir_size_mb = (
    xml_size_mb +
    bin_size_mb
)


print("=" * 80)
print("OPENVINO IR FILES")
print("=" * 80)

print(
    f"XML size : {xml_size_mb:.3f} MB"
)

print(
    f"BIN size : {bin_size_mb:.3f} MB"
)

print(
    f"Total IR : {total_ir_size_mb:.3f} MB"
)

print()


# ============================================================
# 22. RELOAD SAVED OPENVINO IR
# ============================================================

print("=" * 80)
print("RELOADING SAVED IR")
print("=" * 80)

core = ov.Core()

loaded_ov_model = core.read_model(
    model=str(FP32_XML)
)


print(
    "Saved IR successfully reloaded."
)

print()


# ============================================================
# 23. COMPILE OPENVINO CPU MODEL
# ============================================================

print("=" * 80)
print("COMPILING OPENVINO CPU MODEL")
print("=" * 80)

compile_start = time.perf_counter()


compiled_model = core.compile_model(
    loaded_ov_model,
    "CPU"
)


compile_time = (
    time.perf_counter()
    -
    compile_start
)


print(
    f"Compilation time: "
    f"{compile_time:.3f} seconds"
)

print()


# ============================================================
# 24. OPENVINO INFERENCE
# ============================================================

print("=" * 80)
print("OPENVINO INFERENCE")
print("=" * 80)

input_layer = (
    compiled_model.inputs[0]
)

output_layer = (
    compiled_model.outputs[0]
)


numpy_input = (
    example_input
    .numpy()
)


ov_result = compiled_model(
    {
        input_layer.any_name:
            numpy_input
    }
)


ov_output = np.asarray(
    ov_result[output_layer]
)


print(
    "OpenVINO output shape:",
    ov_output.shape
)

print(
    "OpenVINO output dtype:",
    ov_output.dtype
)

assert ov_output.shape == (
    1,
    4764
)

print()


# ============================================================
# 25. PYTORCH → OPENVINO NUMERICAL VALIDATION
# ============================================================

print("=" * 80)
print("PYTORCH vs OPENVINO VALIDATION")
print("=" * 80)


torch_output_np = (
    torch_output
    .detach()
    .cpu()
    .numpy()
)


difference = (
    torch_output_np -
    ov_output
)


absolute_difference = np.abs(
    difference
)


max_abs_error = float(
    absolute_difference.max()
)

mean_abs_error = float(
    absolute_difference.mean()
)

rmse = float(
    np.sqrt(
        np.mean(
            difference ** 2
        )
    )
)


torch_flat = (
    torch_output_np
    .reshape(-1)
)

ov_flat = (
    ov_output
    .reshape(-1)
)


cosine_similarity = float(
    np.dot(
        torch_flat,
        ov_flat
    )
    /
    (
        np.linalg.norm(torch_flat)
        *
        np.linalg.norm(ov_flat)
        +
        1e-12
    )
)


print(
    f"Maximum absolute error : "
    f"{max_abs_error:.8e}"
)

print(
    f"Mean absolute error    : "
    f"{mean_abs_error:.8e}"
)

print(
    f"RMSE                   : "
    f"{rmse:.8e}"
)

print(
    f"Output cosine similarity: "
    f"{cosine_similarity:.10f}"
)

print()


# ============================================================
# 26. TOP-1 / TOP-5 AGREEMENT
# ============================================================

torch_top1 = int(
    np.argmax(
        torch_output_np,
        axis=1
    )[0]
)

ov_top1 = int(
    np.argmax(
        ov_output,
        axis=1
    )[0]
)


torch_top5 = np.argsort(
    torch_output_np[0]
)[-5:][::-1]


ov_top5 = np.argsort(
    ov_output[0]
)[-5:][::-1]


top1_match = (
    torch_top1 ==
    ov_top1
)


top5_overlap = len(
    set(torch_top5.tolist())
    &
    set(ov_top5.tolist())
)


print("=" * 80)
print("PREDICTION AGREEMENT")
print("=" * 80)

print(
    f"PyTorch Top-1 class : {torch_top1}"
)

print(
    f"OpenVINO Top-1 class: {ov_top1}"
)

print(
    f"Top-1 agreement     : "
    f"{top1_match}"
)

print()

print(
    "PyTorch Top-5:"
)

print(
    torch_top5.tolist()
)

print()

print(
    "OpenVINO Top-5:"
)

print(
    ov_top5.tolist()
)

print()

print(
    f"Top-5 overlap: "
    f"{top5_overlap}/5"
)

print()


# ============================================================
# 27. NUMERICAL VALIDATION CRITERIA
# ============================================================
#
# We use tolerant numerical criteria rather than exact equality
# because framework conversion can introduce tiny floating-point
# differences.
#
# ============================================================

TOP1_PASS = (
    top1_match
)

TOP5_PASS = (
    top5_overlap >= 5
)

COSINE_PASS = (
    cosine_similarity >= 0.9999
)

ERROR_PASS = (
    max_abs_error <= 1e-3
)


print("=" * 80)
print("CONVERSION VALIDATION STATUS")
print("=" * 80)

print(
    f"Top-1 agreement : "
    f"{'PASS' if TOP1_PASS else 'FAIL'}"
)

print(
    f"Top-5 agreement : "
    f"{'PASS' if TOP5_PASS else 'FAIL'}"
)

print(
    f"Cosine >= 0.9999: "
    f"{'PASS' if COSINE_PASS else 'FAIL'}"
)

print(
    f"Max error <= 1e-3: "
    f"{'PASS' if ERROR_PASS else 'FAIL'}"
)

print()


conversion_valid = (
    TOP1_PASS
    and
    TOP5_PASS
    and
    COSINE_PASS
    and
    ERROR_PASS
)


if conversion_valid:

    print(
        "✓ OPENVINO CONVERSION VALIDATED"
    )

else:

    print(
        "⚠ OPENVINO CONVERSION VALIDATION FAILED"
    )

    print()
    print(
        "Do NOT proceed to performance benchmarking "
        "until this discrepancy is investigated."
    )


print()


# ============================================================
# 28. SAVE CONVERSION METADATA
# ============================================================

conversion_metadata = {

    "stage":
        "4.7",

    "title":
        "CISLR OpenVINO Conversion",

    "selected_model":
        "MobileNetV3-Large",

    "selection_source":
        str(stage46_json),

    "pytorch_checkpoint":
        str(CHECKPOINT_PATH),

    "openvino_xml":
        str(FP32_XML),

    "openvino_bin":
        str(FP32_BIN),

    "precision":
        "FP32",

    "input_shape":
        [1, 3, 224, 224],

    "input_dtype":
        "float32",

    "num_classes":
        num_classes,

    "total_parameters":
        int(total_params),

    "trainable_parameters":
        int(trainable_params),

    "conversion_time_seconds":
        float(conversion_time),

    "save_time_seconds":
        float(save_time),

    "compile_time_seconds":
        float(compile_time),

    "pytorch_output_shape":
        list(torch_output_np.shape),

    "openvino_output_shape":
        list(ov_output.shape),

    "validation": {

        "max_absolute_error":
            max_abs_error,

        "mean_absolute_error":
            mean_abs_error,

        "rmse":
            rmse,

        "cosine_similarity":
            cosine_similarity,

        "pytorch_top1":
            torch_top1,

        "openvino_top1":
            ov_top1,

        "top1_match":
            bool(top1_match),

        "pytorch_top5":
            torch_top5.tolist(),

        "openvino_top5":
            ov_top5.tolist(),

        "top5_overlap":
            int(top5_overlap),

        "conversion_valid":
            bool(conversion_valid),
    },

    "files": {

        "xml_size_mb":
            xml_size_mb,

        "bin_size_mb":
            bin_size_mb,

        "total_ir_size_mb":
            total_ir_size_mb,
    },

    "environment": {

        "python":
            sys.version,

        "pytorch":
            torch.__version__,

        "torchvision":
            torchvision.__version__,

        "openvino":
            ov.__version__,

        "platform":
            platform.platform(),
    },

    "protocol": {

        "batch_size":
            1,

        "channels":
            3,

        "height":
            224,

        "width":
            224,

        "device":
            "CPU for validation",

        "conversion_api":
            "openvino.convert_model",

        "example_input_used":
            True,
    },
}


metadata_path = (
    STAGE47_DIR /
    "stage4_7_conversion_metadata.json"
)


with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        conversion_metadata,
        f,
        indent=2
    )


print(
    f"Saved metadata:\n{metadata_path}"
)

print()


# ============================================================
# 29. SAVE VALIDATION CSV
# ============================================================

validation_df = pd.DataFrame(
    [
        {
            "model":
                "MobileNetV3-Large",

            "precision":
                "FP32",

            "input_shape":
                "1x3x224x224",

            "num_classes":
                num_classes,

            "pytorch_top1":
                torch_top1,

            "openvino_top1":
                ov_top1,

            "top1_match":
                top1_match,

            "top5_overlap":
                top5_overlap,

            "max_absolute_error":
                max_abs_error,

            "mean_absolute_error":
                mean_abs_error,

            "rmse":
                rmse,

            "cosine_similarity":
                cosine_similarity,

            "conversion_valid":
                conversion_valid,

            "xml_size_mb":
                xml_size_mb,

            "bin_size_mb":
                bin_size_mb,

            "total_ir_size_mb":
                total_ir_size_mb,

            "conversion_time_seconds":
                conversion_time,

            "compile_time_seconds":
                compile_time,
        }
    ]
)


validation_csv = (
    STAGE47_DIR /
    "stage4_7_conversion_validation.csv"
)


validation_df.to_csv(
    validation_csv,
    index=False
)


print(
    f"Saved validation CSV:\n{validation_csv}"
)

print()


# ============================================================
# 30. CHECKSUM FILES
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


file_hashes = {

    "pytorch_checkpoint":
        sha256_file(
            CHECKPOINT_PATH
        ),

    "openvino_xml":
        sha256_file(
            FP32_XML
        ),

    "openvino_bin":
        sha256_file(
            FP32_BIN
        ),

    "stage4_6_selection":
        sha256_file(
            stage46_json
        ),
}


hash_path = (
    STAGE47_DIR /
    "stage4_7_artifact_hashes.json"
)


with open(
    hash_path,
    "w"
) as f:

    json.dump(
        file_hashes,
        f,
        indent=2
    )


print(
    f"Saved hashes:\n{hash_path}"
)

print()


# ============================================================
# 31. FINAL REPORT
# ============================================================

print()
print("=" * 80)
print("STAGE 4.7 COMPLETE")
print("=" * 80)

print()

print(
    "Selected model:"
)

print(
    "  >>> MobileNetV3-Large"
)

print()

print(
    "OpenVINO precision:"
)

print(
    "  >>> FP32"
)

print()

print(
    "Input:"
)

print(
    "  >>> [1, 3, 224, 224]"
)

print()

print(
    "Output:"
)

print(
    "  >>> [1, 4764]"
)

print()

print(
    "OpenVINO IR:"
)

print(
    f"  XML: {FP32_XML}"
)

print(
    f"  BIN: {FP32_BIN}"
)

print()

print(
    "IR size:"
)

print(
    f"  >>> {total_ir_size_mb:.3f} MB"
)

print()

print(
    "Validation:"
)

print(
    f"  Top-1 agreement       : "
    f"{'PASS' if TOP1_PASS else 'FAIL'}"
)

print(
    f"  Top-5 agreement       : "
    f"{'PASS' if TOP5_PASS else 'FAIL'}"
)

print(
    f"  Cosine similarity     : "
    f"{cosine_similarity:.10f}"
)

print(
    f"  Maximum absolute error: "
    f"{max_abs_error:.8e}"
)

print()

print(
    "Overall conversion:"
)

print(
    f"  >>> "
    f"{'VALIDATED' if conversion_valid else 'FAILED'}"
)

print()

print(
    "Artifacts:"
)

print(
    f"  {metadata_path}"
)

print(
    f"  {validation_csv}"
)

print(
    f"  {hash_path}"
)

print()

print(
    "NEXT:"
)

print(
    "  >>> Stage 4.8 — OpenVINO Benchmark"
)

print()

print(
    "DO NOT RERUN:"
)

print(
    "  4.3 Computational Benchmark"
)

print(
    "  4.4 Model Training"
)

print(
    "  4.5 Official CISLR Evaluation"
)

print(
    "  4.6 Model Comparison & Selection"
)

print("=" * 80)

CISLR — STAGE 4.7 OPENVINO CONVERSION

Project root:
/home/dipshikha/Music/cao/CISLR_RESEARCH

Selected model:
MobileNetV3-Large

Checkpoint:
/home/dipshikha/Music/cao/CISLR_RESEARCH/models/stage4_4/mobilenet_v3_large_v2/mobilenet_v3_large_stage4_4_v2_best.pt

Output:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_7_openvino_conversion

ENVIRONMENT
Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
PyTorch: 2.14.0+cu130
Torchvision: 0.29.0+cu130
OpenVINO: 2026.3.1-22476-759c5a6ab8c-releases/2026/3
Platform: Linux-7.0.0-31-generic-x86_64-with-glibc2.39

Stage 4.6 selected model: MobileNetV3-Large
Stage 4.6 selection verified.

CHECKPOINT
Path : /home/dipshikha/Music/cao/CISLR_RESEARCH/models/stage4_4/mobilenet_v3_large_v2/mobilenet_v3_large_stage4_4_v2_best.pt
Size : 118.284 MB

Loading PyTorch checkpoint...
Checkpoint object type: <class 'dict'>

Checkpoint dictionary keys:
  - stage
  - version
  - model
  - epoch
  - model_state_dict
  - optimizer_state_dict
  - train_

/home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/dipshikha/miniconda3/envs/openvino_face/lib/python3.11/site-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Conversion time: 2.560 seconds

OPENVINO MODEL INPUT
Number of inputs: 1
Input 0:
  Name   : x
  Shape  : [?,3,?,?]
  Type   : <Type: 'float32'>

OPENVINO MODEL OUTPUT
Number of outputs: 1
Output 0:
  Name  : <unnamed>
  Shape : [?,4764]
  Type  : <Type: 'float32'>

OpenVINO output shape: [?,4764]

SAVING OPENVINO FP32 IR
Saved in 0.035 seconds
XML: /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_7_openvino_conversion/fp32/mobilenet_v3_large_stage4_7_fp32.xml
BIN: /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_7_openvino_conversion/fp32/mobilenet_v3_large_stage4_7_fp32.bin

OPENVINO IR FILES
XML size : 0.175 MB
BIN size : 39.263 MB
Total IR : 39.438 MB

RELOADING SAVED IR
Saved IR successfully reloaded.

COMPILING OPENVINO CPU MODEL
Compilation time: 0.178 seconds

OPENVINO INFERENCE
OpenVINO output shape: (1, 4764)
OpenVINO output dtype: float32

PYTORCH vs OPENVINO VALIDATION
Maximum absolute error : 1.33514404e-05
Mean absolute error    : 2.99993303e-06
RMSE            